In [1]:
import torch

SEED = 1234
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# 0. Choosing "teacher" model

In [2]:
checkpoint = "bert-base-uncased"
task_name  = "mnli"

# 1. Loading our mrpc part of the GLUE dataset

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", task_name)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 392702
    })
    validation_matched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 9815
    })
    validation_mismatched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 9832
    })
    test_matched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 9796
    })
    test_mismatched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 9847
    })
})

In [5]:
raw_datasets['train']

Dataset({
    features: ['premise', 'hypothesis', 'label', 'idx'],
    num_rows: 392702
})

In [6]:
from datasets import DatasetDict

In [7]:
raw_datasets = DatasetDict({
    "train": raw_datasets['train'],
    "validation_matched": raw_datasets['validation_matched'],
    "validation_mismatched": raw_datasets['validation_mismatched'],
    "test_matched": raw_datasets['test_matched'],
    "test_mismatched": raw_datasets['test_mismatched']
})

In [8]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 392702
    })
    validation_matched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 9815
    })
    validation_mismatched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 9832
    })
    test_matched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 9796
    })
    test_mismatched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx'],
        num_rows: 9847
    })
})

In [9]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

{'premise': 'Conceptually cream skimming has two basic dimensions - product and geography.',
 'hypothesis': 'Product and geography are what make cream skimming work. ',
 'label': 1,
 'idx': 0}

In [10]:
raw_train_dataset[5]['premise'], raw_train_dataset[5]['hypothesis']

("my walkman broke so i'm upset now i just have to turn the stereo up real loud",
 "I'm upset that my walkman broke and now I have to turn the stereo up really loud.")

In [11]:
raw_train_dataset[5]['label']

0

In [12]:
raw_train_dataset[5]['idx']

5

In [13]:
raw_train_dataset.features

{'premise': Value('string'),
 'hypothesis': Value('string'),
 'label': ClassLabel(names=['entailment', 'neutral', 'contradiction']),
 'idx': Value('int32')}

# 2. Preprocess

In [14]:
def tokenize_function(example):
    return tokenizer(example["premise"], example["hypothesis"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

Map: 100%|██████████| 9847/9847 [00:00<00:00, 47041.24 examples/s]


DatasetDict({
    train: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 392702
    })
    validation_matched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9815
    })
    validation_mismatched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9832
    })
    test_matched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9796
    })
    test_mismatched: Dataset({
        features: ['premise', 'hypothesis', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 9847
    })
})

# 3. Preparing for Training

In [15]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [16]:
tokenized_datasets = tokenized_datasets.remove_columns(["premise", "hypothesis", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [17]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=32, collate_fn=data_collator
)

validation_matched = DataLoader(
    tokenized_datasets["validation_matched"], batch_size=32, collate_fn=data_collator
)

validation_mismatched = DataLoader(
    tokenized_datasets["validation_mismatched"], batch_size=32, collate_fn=data_collator
)

# eval_dataloader = DataLoader(
#     tokenized_datasets["test"], batch_size=32, collate_fn=data_collator
# )

In [18]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([32]),
 'input_ids': torch.Size([32, 83]),
 'token_type_ids': torch.Size([32, 83]),
 'attention_mask': torch.Size([32, 83])}

# 4. Loading model

In [19]:
import sys
sys.path.insert(0, '../')

In [20]:
from Bert_model.modeling_bert import BertForSequenceClassification

In [21]:
# id2label, label2id dicts for the outputs for the model
labels = tokenized_datasets["train"].features["labels"].names
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

In [22]:
model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
model.set_use_module_grafting(False)
model.set_use_scc_status(False)

In [24]:
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)

tensor(1.1568, grad_fn=<NllLossBackward0>) torch.Size([32, 3])


In [25]:
device = torch.device("cuda:1") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

device

device(type='cuda', index=1)

### Load Trained Weights

In [26]:
load_path = '../glue_fine_tune/weights/'
best_weight = torch.load(load_path + f'bert-{task_name}.pt')
model.load_state_dict(best_weight['model_state_dict'])

<All keys matched successfully>

In [27]:
from train_eval_func import eval_loop

In [28]:
eval_loop(model, validation_matched, task_name, device)[0]

{'accuracy': 0.844727457972491}

# 5. Merge Most Similar Layers

In [29]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn
from tqdm import tqdm

In [30]:
import os
from merge_helper import merge_bert_layers, LayerMergeTracker
from train_eval_func import train, get_primary_metric, set_lr_scheduler, iterative_cka_merge_and_train, get_recovery_epoch
from torch.optim import AdamW

In [31]:
init_metric = eval_loop(model, validation_matched, task_name, device)[0]
init_metric

{'accuracy': 0.844727457972491}

In [32]:
tracker = LayerMergeTracker(num_layers=12)

print("Initial state:")
print(f"  {tracker.get_mapping()}")
print(f"  Total layers: {len(tracker)}")
print()

Initial state:
  [[0], [1], [2], [3], [4], [5], [6], [7], [8], [9], [10], [11]]
  Total layers: 12



In [33]:
from CKA import CKAEvaluator

In [34]:
cka_eval = CKAEvaluator(device)

In [35]:
recovery_epochs = get_recovery_epoch(task_name)
recovery_epochs

3

In [36]:
history = iterative_cka_merge_and_train(
    model=model,
    train_dataloader=train_dataloader,
    val_dataloader=validation_matched,
    task_name=task_name,
    device=device,
    cka_evaluator=cka_eval,
    tracker=tracker,
    init_metric=init_metric,
    num_merges=6,
    target_layers=[6, 7, 8],  # Save models with 6, 7, 8 layers
    recovery_epochs=recovery_epochs,
    recovery_lr=1e-5,
    save_dir='./weights/',
    cka_max_iter=115,
    validation_mismatched=validation_mismatched
)

Target Metric for mnli: accuracy
Original accuracy score: 0.8447
Recovery threshold: 0.0084

Merge Iteration: 1/6

[1/5] Computing CKA Similarity...


CKA Evaluation: 100%|██████████| 115/115 [00:14<00:00,  7.72it/s]


  Similarity Stats:
    Average: 0.899983
    Highest: 0.948885
    Lowest:  0.811374

[2/5] Merging Layers...
  Merged Layers: 5 + 6
  Layer Composition: [[0], [1], [2], [3], [4], [5, 6], [7], [8], [9], [10], [11]]
  Total Layers: 11

[3/5] Evaluating Post-Merge Performance...
  Metrics:
    ★ accuracy: 0.8355

[4/5] Recovery Training...
  Performance drop: 0.0093 (threshold: 0.0084)
  → Recovery training NEEDED
  → Training: 3 epochs, lr=1e-05


Training:   0%|          | 0/36816 [00:00<?, ?it/s]/home/abhinavl/work/Layer_Graft/layer-grafting-publication/similar_layer_merge/../train_eval_func.py:223: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:835.)
  total_loss += float(loss)
Training:  33%|███▎      | 12272/36816 [40:07<48:53,  8.37it/s]  

✓ Saved checkpoint (best loss: 0.5837)

<----------------- Epoch 1 ----------------->
Loss: 0.2, Training Metrics:
  accuracy: 0.9310
Validation Loss: 0.58, Validation Metrics:
  accuracy: 0.8328
Elapsed Time: 2432.4252 sec


...

Training:  67%|██████▋   | 24544/36816 [1:16:34<27:59,  7.31it/s]  

✓ Saved checkpoint (best loss: 1.0890)

<----------------- Epoch 2 ----------------->
Loss: 0.1, Training Metrics:
  accuracy: 0.9701
Validation Loss: 1.09, Validation Metrics:
  accuracy: 0.8203
Elapsed Time: 2303.4916 sec


Training: 100%|██████████| 36816/36816 [1:56:10<00:00,  5.28it/s]   

✓ Saved checkpoint (best loss: 1.2337)

<----------------- Epoch 3 ----------------->
Loss: 0.1, Training Metrics:
  accuracy: 0.9754
Validation Loss: 1.23, Validation Metrics:
  accuracy: 0.8235
Elapsed Time: 2331.7021 sec

Total Training Time: 6970.63 sec
Chosen Primary Metric: accuracy
Best Metric Run: 0.8235

  → Loading best checkpoint...


  ✓ Loaded best checkpoint from epoch 3
    Best val loss: 1.2337
    Best val metrics: {'accuracy': 0.8235354049923587}
  Metrics for MNLI Mismatch:
    ★ accuracy: 0.8278
  ✓ Cleaned up temporary checkpoint

[5/5] Saving Model...
  ✓ Saved 6-layer model to ./weights/layer-6-mnli.pt

MERGE LOOP COMPLETED
Total merges: 6
Final layers: 6
Final composition: [[0], [1, 2, 3], [4], [5, 6, 7], [8, 9, 10], [11]]


In [37]:
# See what happened
print("Merge history:")
for i, iteration in enumerate(history['iterations']):
    print(f"  Iteration {iteration['iteration']}:")
    print(f"    Merged: {iteration['merged_layers']}")
    print(f"    Layers after: {iteration['num_layers_after']}")
    print(f"    Composition: {history['layer_compositions'][i]}")
    
    if history['training_stats'][i]:
        final_metrics = history['metrics_after_recovery'][i]
        print(f"    Final metrics: {final_metrics}")

Merge history:
  Iteration 1:
    Merged: (5, 6)
    Layers after: 11
    Composition: [[0], [1, 2, 3], [4], [5, 6, 7], [8, 9, 10], [11]]
    Final metrics: {'accuracy': 0.8369842078451349}
  Iteration 2:
    Merged: (7, 8)
    Layers after: 10
    Composition: [[0], [1, 2, 3], [4], [5, 6, 7], [8, 9, 10], [11]]
    Final metrics: {'accuracy': 0.8310748853795211}
  Iteration 3:
    Merged: (5, 6)
    Layers after: 9
    Composition: [[0], [1, 2, 3], [4], [5, 6, 7], [8, 9, 10], [11]]
    Final metrics: {'accuracy': 0.8265919510952624}
  Iteration 4:
    Merged: (1, 2)
    Layers after: 8
    Composition: [[0], [1, 2, 3], [4], [5, 6, 7], [8, 9, 10], [11]]
    Final metrics: {'accuracy': 0.8249617931737137}
  Iteration 5:
    Merged: (1, 2)
    Layers after: 7
    Composition: [[0], [1, 2, 3], [4], [5, 6, 7], [8, 9, 10], [11]]
    Final metrics: {'accuracy': 0.8230259806418747}
  Iteration 6:
    Merged: (4, 5)
    Layers after: 6
    Composition: [[0], [1, 2, 3], [4], [5, 6, 7], [8, 9, 10